In [ ]:
import os
import json
import operator
import OpenDartReader
import yfinance as yf

from datetime import datetime
from typing import Annotated, List, Literal, Optional
from typing_extensions import TypedDict

from pydantic import BaseModel, Field

from dotenv import load_dotenv
from tavily import TavilyClient
from docling.document_converter import DocumentConverter

from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool
from langchain_core.runnables import RunnableConfig

from langgraph.graph import StateGraph, END
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

from langchain_teddynote import logging
from langchain_teddynote.messages import random_uuid, invoke_graph
from langchain_teddynote.graphs import visualize_graph


# ---------------------------------------------------------
# 0. 환경 설정
# ---------------------------------------------------------
load_dotenv()

logging.langsmith(os.environ["STUDENT_NAME"] + "-" + "Supervisor_Validation_Agent")


# ---------------------------------------------------------
# 1. 사용자 투자 프로파일 로딩
# ---------------------------------------------------------
def get_user_profile():
    with open("./data/Your_profile.json", "r", encoding="utf-8") as f:
        profile_data = json.load(f)

    return json.dumps(profile_data, ensure_ascii=False, indent=2)


# ---------------------------------------------------------
# 2. LLM 설정
# ---------------------------------------------------------


llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.1,  # 라우팅의 정확도와 일관성을 위해 0으로 고정합니다.
    # model="gpt-5.5",  
  
    api_key=os.environ.get("OPENAI_API_KEY"),
)

print("##################### Completed Cell #####################")


In [ ]:

# ---------------------------------------------------------
# 3. 상태(State) 정의
# ---------------------------------------------------------
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], operator.add]
    next: str
    validation_result: str
    retry_count: int
    retry_reason: str


# ---------------------------------------------------------
# 4. Worker Agent 생성 함수
# ---------------------------------------------------------
def create_worker_agent(system_message: str, tools: list = []):
    return create_react_agent(
        model=llm,
        tools=tools,
        state_modifier=SystemMessage(content=system_message),
    )


print("##################### Completed Cell #####################")

In [ ]:

# ---------------------------------------------------------
# 5. 경제데이터 조회 툴
# ---------------------------------------------------------
@tool
def Economy_Data_Tool() -> str:
    """
    투자 환경 판단을 위해 주요 경기 지표를 조회합니다.
    조회 항목: VIX, CLI, 환율, BDI, WTI, 장단기금리차
    """
    file_path = "./data/경제지표_regime_1.pdf"

    if not os.path.exists(file_path):
        return f"오류: 지정된 경로에 '{file_path}' 파일이 존재하지 않습니다."

    try:
        converter = DocumentConverter()
        result = converter.convert(file_path)
        markdown_context = result.document.export_to_markdown()

        if not markdown_context.strip():
            return "파일 파싱은 성공했으나, 추출된 텍스트 컨텍스트가 비어 있습니다."

        return markdown_context

    except Exception as e:
        return f"Docling으로 PDF를 읽는 중 오류가 발생했습니다: {str(e)}"


# ---------------------------------------------------------
# 6. 기업 재무정보 조회 툴
# ---------------------------------------------------------
@tool
def Company_Fin_Tool(company: str, year: Optional[int] = None) -> str:
    """
    OpenDartReader를 사용하여 특정 기업의 주요 재무 지표를 조회합니다.
    조회 항목: 매출액, 영업이익, 당기순이익, 자산총계, 부채총계
    """
    api_key = os.environ["DART_API_KEY"]

    if year is None:
        year = datetime.now().year - 1

    try:
        dart = OpenDartReader(api_key)
        df = dart.finstate(company, year, reprt_code="11011")

        if df is None or df.empty:
            return f"안내: {company}의 {year}년도 연간 사업보고서 데이터가 없거나 기업명을 찾을 수 없습니다."

        financial_data = {
            "매출액": "데이터 없음",
            "영업이익": "데이터 없음",
            "당기순이익": "데이터 없음",
            "자산총계": "데이터 없음",
            "부채총계": "데이터 없음",
        }

        actual_corp_name = company

        for _, row in df.iterrows():
            actual_corp_name = row.get("corp_name", company)
            sj_div = row.get("sj_div", "")
            account_nm = str(row.get("account_nm", "")).replace(" ", "")
            amount_str = row.get("thstrm_amount", "0")

            try:
                formatted_amount = f"{int(str(amount_str).replace(',', '')):,} 원"
            except ValueError:
                formatted_amount = f"{amount_str} 원"

            if sj_div == "BS":
                if "자산총계" in account_nm:
                    financial_data["자산총계"] = formatted_amount
                elif "부채총계" in account_nm:
                    financial_data["부채총계"] = formatted_amount

            elif sj_div in ["IS", "CIS"]:
                if "매출액" in account_nm or account_nm == "매출":
                    financial_data["매출액"] = formatted_amount
                elif "영업이익" in account_nm:
                    financial_data["영업이익"] = formatted_amount
                elif "당기순이익" in account_nm:
                    financial_data["당기순이익"] = formatted_amount

        return (
            f"### [{actual_corp_name}] {year}년도 정기 사업보고서 주요 재무지표\n"
            f"- **조회 대상 기업**: {company}\n\n"
            f"#### 손익계산서\n"
            f"- **연간 매출액**: {financial_data['매출액']}\n"
            f"- **연간 영업이익**: {financial_data['영업이익']}\n"
            f"- **연간 당기순이익**: {financial_data['당기순이익']}\n\n"
            f"#### 재무상태표\n"
            f"- **자산 총계**: {financial_data['자산총계']}\n"
            f"- **부채 총계**: {financial_data['부채총계']}\n"
        )

    except Exception as e:
        return f"OpenDartReader 실행 중 오류가 발생했습니다: {str(e)}"


# ---------------------------------------------------------
# 7. 주식정보 조회 툴
# ---------------------------------------------------------
@tool
def Stock_Data_Tool(ticker: str) -> str:
    """
    Yahoo Finance에서 특정 주식 종목의 시세 및 핵심 투자 지표를 조회합니다.
    예: 삼성전자 005930.KS, SK하이닉스 000660.KS
    """
    try:
        stock = yf.Ticker(ticker.strip().upper())
        info = stock.info

        if not info or "symbol" not in info:
            return f"오류: 티커 '{ticker}'에 해당하는 주식 정보를 찾을 수 없습니다."

        comp_name = info.get("longName", info.get("shortName", ticker))
        currency = info.get("currency", "USD")
        current_price = info.get(
            "currentPrice", info.get("regularMarketPrice", "데이터 없음")
        )

        market_cap = info.get("marketCap", 0)
        trailing_per = info.get("trailingPE", "데이터 없음")
        forward_per = info.get("forwardPE", "데이터 없음")
        price_to_book = info.get("priceToBook", "데이터 없음")
        dividend_yield = (
            info.get("dividendYield", 0) * 100 if info.get("dividendYield") else 0
        )

        formatted_market_cap = f"{market_cap:,}" if market_cap else "데이터 없음"

        return (
            f"### [{comp_name} ({info.get('symbol')})] 야후 파이낸스 투자 지표 분석\n\n"
            f"#### 시세 정보\n"
            f"- **현재가**: {current_price} {currency}\n"
            f"- **시가총액**: {formatted_market_cap} {currency}\n\n"
            f"#### 밸류에이션 지표\n"
            f"- **트레일링 PER**: {trailing_per}\n"
            f"- **포워드 PER**: {forward_per}\n"
            f"- **PBR**: {price_to_book}\n"
            f"- **배당수익률**: {dividend_yield:.2f}%\n"
        )

    except Exception as e:
        return f"야후 파이낸스 조회 중 오류가 발생했습니다: {str(e)}"


# ---------------------------------------------------------
# 8. 뉴스정보 조회 툴
# ---------------------------------------------------------
@tool
def News_Data_Tool(query: str) -> str:
    """
    Tavily Search API를 사용하여 최신 경제, 투자, 기업 뉴스를 검색합니다.
    """
    try:
        tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

        response = tavily_client.search(
            query=query.strip(),
            topic="news",
            max_results=5,
            include_answer=False,
        )

        results = response.get("results", [])

        if not results:
            return f"안내: Tavily 뉴스 검색에서 '{query}' 관련 결과를 찾을 수 없습니다."

        current_date = datetime.now().strftime("%Y-%m-%d")
        result_summary = f"### '{query}' 관련 최신 뉴스 검색 결과 (검색일: {current_date})\n\n"

        for i, res in enumerate(results):
            title = res.get("title", "뉴스 기사")
            url = res.get("url", "#")
            content = res.get("content", "내용 요약 정보가 없습니다.")
            pub_date = res.get("published_date", "최근 발행")

            result_summary += f"{i + 1}. **[{title}]({url})**\n"
            result_summary += f"   - 발행일: {pub_date}\n"
            result_summary += f"   - 요약: {content}\n\n"

        return result_summary

    except Exception as e:
        return f"Tavily 실행 중 오류가 발생했습니다: {str(e)}"


print("##################### Completed Cell #####################")

In [ ]:

# ---------------------------------------------------------
# 9. Worker Agent 생성
# ---------------------------------------------------------
worker_1 = create_worker_agent(
    """
    당신은 거시경제 시장 분석 전문가입니다.
    VIX, CLI, 환율, BDI, WTI, 장단기금리차 등 글로벌 경기 지표를 분석하여
    현재 투자 환경과 시장 국면을 평가하세요.

    [사용자 프로파일]
    """ + get_user_profile()[1:-1],
    [Economy_Data_Tool],
)

worker_2 = create_worker_agent(
    """
    당신은 기업정보 분석 전문가입니다.
    기업의 비즈니스 모델, 재무제표, 매출, 영업이익, 수익성, 안정성,
    경쟁사 대비 강점과 약점을 분석하세요.

    [사용자 프로파일]
    """ + get_user_profile()[1:-1],
    [Company_Fin_Tool],
)

worker_3 = create_worker_agent(
    """
    당신은 기술적 및 계량적 주가 분석 전문가입니다.
    주가 추세, 거래량, 이동평균선, PER, PBR, 밸류에이션, 매매 타이밍을 분석하세요.

    [사용자 프로파일]
    """ + get_user_profile()[1:-1],
    [Stock_Data_Tool],
)

worker_4 = create_worker_agent(
    """
    당신은 경제 뉴스 분석 전문가입니다.
    관심 섹터, 기업, 종목의 최신 뉴스를 분석하여 투자 관점의 이슈,
    이벤트, 호재, 악재, 리스크를 도출하세요.

    [사용자 프로파일]
    """ + get_user_profile()[1:-1],
    [News_Data_Tool],
)

worker_5 = create_worker_agent(
    """
    당신은 투자전략 보고서 작성 전문가입니다. 
    앞선 매크로 시장 분석, 기업정보 분석, 주가 분석, 뉴스 분석 등을 종합하여 
    국내외 마켓 상황 및 리스크 요인, 명확한 투자 의견(매수/매도/홀딩)이 포함된 
    '최종 투자전략 보고서'를 아래 '사용자 프로파일' 정보도 고려해 완성도 높게 작성하세요.

    [사용자 프로파일]
    """ + get_user_profile()[1:-1]
   
)


print("##################### Completed Cell #####################")

In [ ]:

# ---------------------------------------------------------
# 10. Worker 실행 노드
# ---------------------------------------------------------
def run_worker_node(agent, name: str):
    def node(state: AgentState):
        result = agent.invoke(state)
        final_message = result["messages"][-1]

        return {
            "messages": [
                AIMessage(
                    content=f"[{name}의 분석 결과]:\n{final_message.content}",
                    name=name,
                )
            ]
        }

    return node


print("##################### Completed Cell #####################")

In [ ]:

# ---------------------------------------------------------
# 11. Fallback 노드
# ---------------------------------------------------------
def fallback_node(state: AgentState):
    return {
        "messages": [
            AIMessage(
                content=(
                    "죄송합니다. 저는 주식투자 자문 및 자산 분석을 수행하는 "
                    "'AI 투자 메이트' 에이전트입니다. "
                    "경제, 주식, 기업, 뉴스 분석과 관련된 질문을 입력해 주세요."
                ),
                name="Fallback",
            )
        ],
        "next": "FINISH",
    }


# ---------------------------------------------------------
# 12. Supervisor Router
# ---------------------------------------------------------
class SupervisorRouter(BaseModel):
    next: Literal[
        "Economy_Analysis_Expert",
        "Company_Analysis_Expert",
        "Stock_Analysis_Expert",
        "News_Analysis_Expert",
        "Invest_Analysis_Expert",
        "Report_Validation_Expert",
        "Fallback",
        "FINISH",
    ] = Field(description="다음에 실행할 노드")

print("##################### Completed Cell #####################")

In [ ]:


# ---------------------------------------------------------
# 13. Supervisor 노드
# ---------------------------------------------------------
def supervisor_node(state: AgentState):
    messages = state["messages"]

    supervisor_prompt = """
    당신은 주식투자 자문 시스템의 총괄 리더인 'AI 투자 메이트 매니저'입니다.
    사용자의 투자 관련 요청을 파악하고, 아래 6명의 전문가들을 지휘하세요.
    
    [전문가 목록]
    1. Economy_Analysis_Expert:
       매크로 분석. VIX, CLI, 환율, BDI, WTI, 장단기금리차 등 분석

    2. Company_Analysis_Expert:
       기업분석. 재무제표, 사업모델, 펀더멘탈 분석

    3. Stock_Analysis_Expert:
       주가분석. 차트, 추세, 밸류에이션, 매매 타이밍 분석

    4. News_Analysis_Expert:
       뉴스분석. 최신 뉴스, 이벤트, 호재/악재, 리스크 분석

    5. Invest_Analysis_Expert:
       위 분석 결과를 종합하여 최종 투자전략 보고서 작성

    6. Report_Validation_Expert:
       최종 투자전략 보고서 품질 검증 
       
    [라우팅 규칙]
    - 주식, 경제, 기업, 투자, 뉴스 분석과 무관한 질문이면 Fallback을 선택하세요.
    - 투자 분석 요청이면 일반적으로 아래 순서로 진행하세요.
      Economy_Analysis_Expert
      → Company_Analysis_Expert
      → Stock_Analysis_Expert
      → News_Analysis_Expert
      → Invest_Analysis_Expert
      → Report_Validation_Expert

    - 전문가들의 분석 결과가 모순이 있고, 논리적이지 않다면 이전 Worker를 다시 수행 하세요.
    - Invest_Analysis_Expert가 최종 보고서를 작성한 뒤에는 절대 바로 FINISH하지 말고,
      반드시 Report_Validation_Expert로 넘기세요.
    - Report_Validation_Expert가 PASS라고 판단하면 FINISH하세요.
    - Report_Validation_Expert가 RETRY라고 판단하면 retry_target에 해당하는 Worker를 다시 실행하세요.
    - 재실행 후에는 다시 Invest_Analysis_Expert가 보고서를 재작성하도록 하세요.
    """

 

    structured_llm = llm.with_structured_output(SupervisorRouter)

    try:
        response = structured_llm.invoke(
            [SystemMessage(content=supervisor_prompt)] + messages
        )
        return {"next": response.next}

    except Exception as e:
        print(f"[Supervisor 오류]: {e}")
        return {"next": "Fallback"}


print("##################### Completed Cell #####################")

In [ ]:

# ---------------------------------------------------------
# 14. Report Validation Router
# ---------------------------------------------------------
class ValidationRouter(BaseModel):
    validation_result: Literal["PASS", "RETRY"] = Field(
        description="최종 보고서가 충분하면 PASS, 문제가 있으면 RETRY"
    )
    reason: str = Field(description="검증 판단 사유")
    retry_target: Literal[
        "Economy_Analysis_Expert",
        "Company_Analysis_Expert",
        "Stock_Analysis_Expert",
        "News_Analysis_Expert",
        "Invest_Analysis_Expert",
    ] = Field(description="재실행이 필요한 Worker")


# ---------------------------------------------------------
# 15. 최종 보고서 검증 노드
# ---------------------------------------------------------
def report_validation_node(state: AgentState):
    messages = state["messages"]
    retry_count = state.get("retry_count", 0)


    validation_prompt = """
    당신은 최종 투자전략 보고서를 검증하는 품질관리 에이전트입니다.

    [검증 대상]
    - Invest_Analysis_Expert가 작성한 최종 투자전략 보고서
    - 그 이전 Worker들의 분석 결과와 최종 보고서의 일관성

    [검증 기준]
    1. 사용자 질문에 직접 답변했는가?
    2. 투자 근거와 리스크가 함께 제시되었는가?
    3. 사용자 프로파일 및 투자성향 정보가 반영되었는가?
    4. 앞선 Worker 분석과 최종 보고서 사이에 논리적 모순이 없는가?
 
    [판단 규칙]
    - 충분하면 validation_result = PASS
    - 중요한 누락, 모순, 투자 의견 부재, 사용자 질문 미충족이면 validation_result = RETRY
    - RETRY인 경우 retry_target을 지정하세요.
    - 분석 데이터 자체가 정략적으로 부족하면 해당 분석 Worker를 지정하세요.
    """


    """
    ############################################################
    좀 더 센거!!!
    [검증 기준]
    1. 사용자 질문에 직접 답변했는가?
    2. 투자 의견이 명확한가? 매수 / 매도 / 홀딩 중 하나가 있는가?
    3. 매크로, 기업, 주가, 뉴스 분석 결과가 종합 반영되었는가?
    4. 투자 근거와 리스크가 함께 제시되었는가?
    5. 사용자 투자성향이 반영되었는가?
    6. 앞선 Worker 분석과 최종 보고서 사이에 논리적 모순이 없는가?
    7. 데이터 부족 또는 불확실성이 명확히 표시되었는가?
    8. 과도하게 단정적인 투자 권유를 하지 않았는가?
    ###########################################################
    """

    structured_llm = llm.with_structured_output(ValidationRouter)

    try:
        response = structured_llm.invoke(
            [SystemMessage(content=validation_prompt)] + messages
        )

        if response.validation_result == "PASS":
            return {
                "validation_result": "PASS",
                "next": "FINISH",
                "retry_reason": response.reason,
                "messages": [
                    AIMessage(
                        content=(
                            "[Report_Validation_Expert 검증 결과]: PASS\n"
                            f"사유: {response.reason}"
                        ),
                        name="Report_Validation_Expert",
                    )
                ],
            }

        if retry_count >= 2:
            return {
                "validation_result": "PASS",
                "next": "FINISH",
                "retry_reason": response.reason,
                "messages": [
                    AIMessage(
                        content=(
                            "[Report_Validation_Expert 검증 결과]: RETRY 필요\n"
                            f"사유: {response.reason}\n"
                            "하지만 최대 재실행 횟수 2회를 초과하여 현재 보고서를 최종 결과로 종료합니다."
                        ),
                        name="Report_Validation_Expert",
                    )
                ],
            }

        return {
            "validation_result": "RETRY",
            "next": response.retry_target,
            "retry_count": retry_count + 1,
            "retry_reason": response.reason,
            "messages": [
                AIMessage(
                    content=(
                        "[Report_Validation_Expert 검증 결과]: RETRY\n"
                        f"사유: {response.reason}\n"
                        f"재실행 대상: {response.retry_target}\n"
                        f"재실행 횟수: {retry_count + 1}/2"
                    ),
                    name="Report_Validation_Expert",
                )
            ],
        }

    except Exception as e:
        return {
            "validation_result": "PASS",
            "next": "FINISH",
            "retry_reason": str(e),
            "messages": [
                AIMessage(
                    content=(
                        "[Report_Validation_Expert 오류]\n"
                        f"{e}\n"
                        "검증 오류로 인해 안전하게 종료합니다."
                    ),
                    name="Report_Validation_Expert",
                )
            ],
        }


print("##################### Completed Cell #####################")

In [ ]:

# ---------------------------------------------------------
# 16. 그래프 정의
# ---------------------------------------------------------
workflow = StateGraph(AgentState)

workflow.add_node("Supervisor", supervisor_node)

workflow.add_node(
    "Economy_Analysis_Expert",
    run_worker_node(worker_1, "Economy_Analysis_Expert"),
)

workflow.add_node(
    "Company_Analysis_Expert",
    run_worker_node(worker_2, "Company_Analysis_Expert"),
)

workflow.add_node(
    "Stock_Analysis_Expert",
    run_worker_node(worker_3, "Stock_Analysis_Expert"),
)

workflow.add_node(
    "News_Analysis_Expert",
    run_worker_node(worker_4, "News_Analysis_Expert"),
)

workflow.add_node(
    "Invest_Analysis_Expert",
    run_worker_node(worker_5, "Invest_Analysis_Expert"),
)

workflow.add_node("Report_Validation_Expert", report_validation_node)

workflow.add_node("Fallback", fallback_node)


# ---------------------------------------------------------
# 17. 진입점
# ---------------------------------------------------------
workflow.set_entry_point("Supervisor")


# ---------------------------------------------------------
# 18. Supervisor 조건부 엣지
# ---------------------------------------------------------
workflow.add_conditional_edges(
    "Supervisor",
    lambda state: state["next"],
    {
        "Economy_Analysis_Expert": "Economy_Analysis_Expert",
        "Company_Analysis_Expert": "Company_Analysis_Expert",
        "Stock_Analysis_Expert": "Stock_Analysis_Expert",
        "News_Analysis_Expert": "News_Analysis_Expert",
        "Invest_Analysis_Expert": "Invest_Analysis_Expert",
        "Report_Validation_Expert": "Report_Validation_Expert",
        "Fallback": "Fallback",
        "FINISH": END,
    },
)


# ---------------------------------------------------------
# 19. Worker 완료 후 Supervisor 복귀
# ---------------------------------------------------------
workflow.add_edge("Economy_Analysis_Expert", "Supervisor")
workflow.add_edge("Company_Analysis_Expert", "Supervisor")
workflow.add_edge("Stock_Analysis_Expert", "Supervisor")
workflow.add_edge("News_Analysis_Expert", "Supervisor")

# 최종 보고서 작성 후에는 Supervisor가 아니라 검증 Agent로 직접 이동
workflow.add_edge("Invest_Analysis_Expert", "Report_Validation_Expert")


# ---------------------------------------------------------
# 20. 검증 결과에 따른 조건부 엣지
# ---------------------------------------------------------
workflow.add_conditional_edges(
    "Report_Validation_Expert",
    lambda state: state["next"],
    {
        "Economy_Analysis_Expert": "Economy_Analysis_Expert",
        "Company_Analysis_Expert": "Company_Analysis_Expert",
        "Stock_Analysis_Expert": "Stock_Analysis_Expert",
        "News_Analysis_Expert": "News_Analysis_Expert",
        "Invest_Analysis_Expert": "Invest_Analysis_Expert",
        "FINISH": END,
    },
)

workflow.add_edge("Fallback", END)

print("##################### Completed Cell #####################")

In [ ]:


# ---------------------------------------------------------
# 21. 그래프 컴파일 및 시각화
# ---------------------------------------------------------
memory = MemorySaver()

graph = workflow.compile(checkpointer=memory)

visualize_graph(graph)


print("##################### Completed Cell #####################")

In [ ]:

# ---------------------------------------------------------
# 22. 실행 예시 1
# ---------------------------------------------------------
config = RunnableConfig(
    recursion_limit=25,
    configurable={"thread_id": random_uuid()},
)

inputs = {
    "messages": [
        HumanMessage(content="SK하이닉스 주식 매도 타이밍 분석해줘.")
    ],
    "next": "",
    "validation_result": "",
    "retry_count": 0,
    "retry_reason": "",
}

invoke_graph(graph, inputs, config)

